In [9]:
import pandas as pd

df= pd.read_csv("Delivery_Logistics_1.csv")

# Use delivery_status for actual delay analysis; failed orders are counted separately.
df["is_delayed"] = (df["delivery_status"].astype(str).str.strip().str.lower() == "delayed").astype(int)
print("dataset shape:",df.shape)
print("\ncolumn names:",df.columns)
print("\ndataset information")
df.info()


dataset shape: (25000, 16)

column names: Index(['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type',
       'delivery_mode', 'region', 'weather_condition', 'distance_km',
       'package_weight_kg', 'delivery_time_hours', 'expected_time_hours',
       'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost',
       'is_delayed'],
      dtype='str')

dataset information
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   delivery_id          25000 non-null  int64  
 1   delivery_partner     25000 non-null  str    
 2   package_type         25000 non-null  str    
 3   vehicle_type         25000 non-null  str    
 4   delivery_mode        25000 non-null  str    
 5   region               25000 non-null  str    
 6   weather_condition    25000 non-null  str    
 7   distance_km          25000 non-null  float64
 8   pa

In [10]:
df.isnull().sum()
(df.isnull().sum() / len(df))*100

delivery_id            0.0
delivery_partner       0.0
package_type           0.0
vehicle_type           0.0
delivery_mode          0.0
region                 0.0
weather_condition      0.0
distance_km            0.0
package_weight_kg      0.0
delivery_time_hours    0.0
expected_time_hours    0.0
delayed                0.0
delivery_status        0.0
delivery_rating        0.0
delivery_cost          0.0
is_delayed             0.0
dtype: float64

In [3]:
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
24995    False
24996    False
24997    False
24998    False
24999    False
Length: 25000, dtype: bool

In [11]:
df.dtypes

delivery_id              int64
delivery_partner           str
package_type               str
vehicle_type               str
delivery_mode              str
region                     str
weather_condition          str
distance_km            float64
package_weight_kg      float64
delivery_time_hours      int64
expected_time_hours      int64
delayed                    str
delivery_status            str
delivery_rating          int64
delivery_cost          float64
is_delayed               int64
dtype: object

In [5]:
df.describe()

,delivery_id,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delivery_rating,delivery_cost,is_delayed
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,12500.500000,150.390436,25.145898,6.248040,13.107680,3.666000,864.944579,0.213640
std,7217.022701,86.409745,14.368663,3.140935,7.559024,1.149964,435.712593,0.409884
min,1.000000,3.600000,0.670000,0.000000,2.000000,1.000000,95.667400,0.000000
25%,6250.750000,75.900000,12.680000,4.000000,8.000000,3.000000,490.800000,0.000000
50%,12500.500000,151.000000,25.145000,6.000000,8.000000,4.000000,867.535000,0.000000
75%,18750.250000,224.900000,37.660000,8.000000,16.000000,5.000000,1237.910000,0.000000
max,25000.000000,297.100000,49.520000,19.000000,24.000000,5.000000,1632.720600,1.000000


In [12]:
Q1 = df['distance_km'].quantile(0.25)
Q3 = df['distance_km'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

outliers = df[
    (df['distance_km'] < lower_limit) |
    (df['distance_km'] > upper_limit)
]

print("Total Outliers:", len(outliers))

Lower Limit: -147.6
Upper Limit: 448.4
Total Outliers: 0


In [13]:
for col in ['package_weight_kg', 'delivery_rating', 'delivery_cost']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ((df[col] < lower) | (df[col] > upper)).sum()

    print(col, ":", count, "outliers")

package_weight_kg : 0 outliers
delivery_rating : 0 outliers
delivery_cost : 0 outliers


In [14]:
df.head(10)

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost,is_delayed
0,1,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,8,8,no,delivered,3,1632.7206,0
1,2,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,2,3,no,delivered,5,640.1700,0
2,3,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,10,16,no,delivered,4,1448.1700,0
3,4,dhl,electronics,ev van,same day,east,cold,269.7,12.69,6,8,no,delivered,3,1486.5700,0
4,5,dhl,clothing,van,two day,north,foggy,256.7,37.02,9,16,no,delivered,4,1394.5600,0
5,6,amazon logistics,documents,ev bike,express,west,rainy,48.4,33.15,4,2,yes,delayed,3,391.4500,1
6,7,delhivery,groceries,scooter,same day,central,clear,198.3,43.79,6,8,no,delivered,3,1222.8700,0
7,8,xpressbees,fragile items,van,same day,north,cold,114.6,42.63,4,8,no,delivered,3,800.8900,0
8,9,blue dart,clothing,van,same day,south,hot,142.4,14.06,5,8,no,delivered,5,854.1800,0
9,10,delhivery,pharmacy,truck,same day,east,foggy,47.1,29.28,3,8,no,delivered,5,423.3400,0


In [9]:
print(df[['delivery_id', 'delivery_time_hours', 'expected_time_hours']].head(10).to_string(index=False))

 delivery_id  delivery_time_hours  expected_time_hours
           1                    8                    8
           2                    2                    3
           3                   10                   16
           4                    6                    8
           5                    9                   16
           6                    4                    2
           7                    6                    8
           8                    4                    8
           9                    5                    8
          10                    3                    8


In [10]:
(df['delivery_id'] == 250.99).sum()

np.int64(0)

In [11]:
df[['delivery_id']].iloc[240:260]

,delivery_id
240,241
241,242
242,243
243,244
244,245
245,246
246,247
247,248
248,249
249,250


In [15]:
df.iloc[:5].T

,0,1,2,3,4
delivery_id,1,2,3,4,5
delivery_partner,delhivery,xpressbees,shadowfax,dhl,dhl
package_type,automobile parts,cosmetics,groceries,electronics,clothing
vehicle_type,bike,ev van,truck,ev van,van
delivery_mode,same day,express,two day,same day,two day
region,west,central,east,east,north
weather_condition,clear,cold,rainy,cold,foggy
distance_km,297.0,89.6,273.5,269.7,256.7
package_weight_kg,46.96,47.39,26.89,12.69,37.02
delivery_time_hours,8,2,10,6,9


In [16]:
print(df['delivery_time_hours'].head(10).tolist())
print(df['expected_time_hours'].head(10).tolist())

[8, 2, 10, 6, 9, 4, 6, 4, 5, 3]
[8, 3, 16, 8, 16, 2, 8, 8, 8, 8]


In [17]:
for col in ['package_weight_kg', 'delivery_rating', 'delivery_cost']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    count = ((df[col] < lower) | (df[col] > upper)).sum()

    print(col, ":", count, "outliers")
    

package_weight_kg : 0 outliers
delivery_rating : 0 outliers
delivery_cost : 0 outliers


In [15]:
print("Duplicate Delivery IDs:", df['delivery_id'].duplicated().sum())

print("\nDelivery Time Data Type:", df['delivery_time_hours'].dtype)
print("Expected Time Data Type:", df['expected_time_hours'].dtype)

df[['delivery_id', 'delivery_time_hours', 'expected_time_hours']].head(10)

Duplicate Delivery IDs: 0

Delivery Time Data Type: int64
Expected Time Data Type: int64


,delivery_id,delivery_time_hours,expected_time_hours
0,1,8,8
1,2,2,3
2,3,10,16
3,4,6,8
4,5,9,16
5,6,4,2
6,7,6,8
7,8,4,8
8,9,5,8
9,10,3,8


In [18]:
print("Dataset Shape:", df.shape)

print("\nMissing Values:", df.isnull().sum().sum())

print("Duplicate Rows:", df.duplicated().sum())

print("Unique Delivery IDs:", df['delivery_id'].nunique())

print("\nData Types:")
print(df.dtypes)

Dataset Shape: (25000, 16)

Missing Values: 0
Duplicate Rows: 0
Unique Delivery IDs: 25000

Data Types:
delivery_id              int64
delivery_partner           str
package_type               str
vehicle_type               str
delivery_mode              str
region                     str
weather_condition          str
distance_km            float64
package_weight_kg      float64
delivery_time_hours      int64
expected_time_hours      int64
delayed                    str
delivery_status            str
delivery_rating          int64
delivery_cost          float64
is_delayed               int64
dtype: object


In [20]:
df = pd.read_csv("Delivery_Logistics_1.csv")

# The original delayed column marks both delayed and failed orders as "yes".
# For delay-only KPIs, use delivery_status == "delayed".
df["is_delayed"] = (df["delivery_status"].astype(str).str.strip().str.lower() == "delayed").astype(int)

print("Original delayed flag (includes failed orders):")
print(df["delayed"].value_counts(dropna=False))
print("\nActual delay flag (delayed status only):")
print(df["is_delayed"].value_counts(dropna=False))

Original delayed flag (includes failed orders):
delayed
no     18331
yes     6669
Name: count, dtype: int64

Actual delay flag (delayed status only):
is_delayed
0    19659
1     5341
Name: count, dtype: int64


In [21]:
avg_time = df['delivery_time_hours'].mean()

delay_rate = df['is_delayed'].mean() * 100

avg_cost = df['delivery_cost'].mean()

print("Average Delivery Time:", round(avg_time, 2), "hours")
print("Actual Delay Rate (failed orders excluded):", round(delay_rate, 2), "%")
print("Average Delivery Cost:", round(avg_cost, 2))

Average Delivery Time: 6.25 hours
Actual Delay Rate (failed orders excluded): 21.36 %
Average Delivery Cost: 864.94


In [22]:
region_delay = df.groupby('region')['is_delayed'].mean() * 100

print(region_delay.sort_values(ascending=False).round(2))

region
north      21.80
central    21.60
west       21.32
south      21.28
east       20.82
Name: is_delayed, dtype: float64


In [23]:
weather_delay = df.groupby('weather_condition')['is_delayed'].mean() * 100

print(weather_delay.sort_values(ascending=False).round(2))


weather_condition
stormy    32.54
rainy     29.97
foggy     24.39
clear     14.26
hot       13.87
cold      12.87
Name: is_delayed, dtype: float64


In [21]:
mode_delay = df.groupby('delivery_mode')['is_delayed'].mean() * 100

print(mode_delay.sort_values(ascending=False).round(2))

delivery_mode
express     59.36
same day    25.78
two day      0.35
standard     0.00
Name: is_delayed, dtype: float64


In [24]:
vehicle_delay = df.groupby('vehicle_type')['is_delayed'].mean() * 100

print(vehicle_delay.sort_values(ascending=False).round(2))

vehicle_type
bike       21.51
ev van     21.40
van        21.38
scooter    21.35
ev bike    21.31
truck      21.23
Name: is_delayed, dtype: float64


In [23]:
correlation = df['distance_km'].corr(df['delivery_time_hours'])

print("Distance-Time Correlation:", round(correlation, 2))

Distance-Time Correlation: 0.69


In [25]:
package_delay = df.groupby('package_type')['is_delayed'].mean() * 100

print(package_delay.sort_values(ascending=False).round(2))

package_type
pharmacy            22.49
groceries           22.21
electronics         22.17
documents           21.78
cosmetics           21.65
fragile items       21.42
clothing            20.60
automobile parts    20.43
furniture           19.52
Name: is_delayed, dtype: float64


In [25]:
partner_delay = df.groupby('delivery_partner')['is_delayed'].mean() * 100

print(partner_delay.sort_values(ascending=False).round(2))

delivery_partner
shadowfax           22.44
xpressbees          22.36
ecom express        22.23
ekart               21.92
amazon logistics    21.58
blue dart           21.16
dhl                 20.41
fedex               20.40
delhivery           19.81
Name: is_delayed, dtype: float64


In [26]:
rating_analysis = df.groupby('delivery_status')['delivery_rating'].mean()

print(rating_analysis.round(2))

delivery_status
delayed      2.40
delivered    4.21
failed       1.31
Name: delivery_rating, dtype: float64


In [27]:
print(df['delivery_status'].value_counts())

delivery_status
delivered    18331
delayed       5341
failed        1328
Name: count, dtype: int64


In [28]:
print(pd.crosstab(df['delivery_status'], df['is_delayed']))

is_delayed           0     1
delivery_status             
delayed              0  5341
delivered        18331     0
failed            1328     0


In [29]:
total_deliveries = len(df)
delayed_orders = (df['delivery_status'] == 'delayed').sum()
delay_rate = (delayed_orders / total_deliveries) * 100

print("Actual Delay Rate:", round(delay_rate, 2), "%")

Actual Delay Rate: 21.36 %


In [30]:
cost_by_region = df.groupby('region')['delivery_cost'].mean()

print(cost_by_region.round(2))

region
central    869.95
east       861.46
north      860.44
south      858.30
west       874.21
Name: delivery_cost, dtype: float64


In [31]:
time_by_vehicle = df.groupby('vehicle_type')['delivery_time_hours'].mean()

print(time_by_vehicle.round(2))

vehicle_type
bike       6.28
ev bike    6.20
ev van     6.25
scooter    6.28
truck      6.22
van        6.27
Name: delivery_time_hours, dtype: float64


In [32]:
correlation = df['distance_km'].corr(df['delivery_cost'])
print("Distance-Cost Correlation:", round(correlation, 2))

Distance-Cost Correlation: 0.99


In [33]:
df['time_difference'] = (
    df['delivery_time_hours'] - df['expected_time_hours']
)

print("Average Time Difference:",
      round(df['time_difference'].mean(), 2), "hours")

print("Deliveries Taking Longer:",
      (df['time_difference'] > 0).sum())

print("Deliveries Taking Less Time:",
      (df['time_difference'] < 0).sum())

Average Time Difference: -6.86 hours
Deliveries Taking Longer: 5466
Deliveries Taking Less Time: 17840
